# Gerar Ground Truth de PII via Gemini

Varre os documentos reais em `data/documento_social/01.arquivos_originais/arquivos_reais/`, envia cada um ao Gemini usando o prompt e o schema estruturado em `prompts/`, e salva o gabarito (JSON por documento) em `data/documento_social/02.gabarito_gemini/<modelo>/` — uma subpasta por modelo (ex.: `02.gabarito_gemini/gemini-2.5-pro/`, `02.gabarito_gemini/gemini-2.5-flash-lite/`), para poder comparar gabaritos de modelos diferentes lado a lado sem um sobrescrever o outro.

Esse gabarito serve de entrada para `notebooks/03.align_ground_truth.ipynb`, que depois casa texto+contexto com bounding boxes reais extraídas pelo pipeline local (ver PRD.md, seções 9 e 10).

**Documentos sintéticos não passam por aqui** — ver `notebooks/00.gerar_documentos_sinteticos.ipynb`, que gera os PDFs em `01.arquivos_originais/arquivos_sinteticos/` já com gabarito conhecido (sem precisar de LLM).

**Pré-requisitos:**
- `pip install google-genai python-dotenv`
- Variável de ambiente `GEMINI_API_KEY` definida (ou um arquivo `.env` na raiz do projeto com `GEMINI_API_KEY=...`)

In [2]:
import datetime
import json
import re
import shutil
import tempfile
import time
import unicodedata
from pathlib import Path

from google import genai
from google.genai import types

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

## Configuração de caminhos

Resolve a raiz do projeto automaticamente (procura `requirements.txt` subindo a partir do diretório atual), para o notebook funcionar independentemente de ser executado a partir de `notebooks/` ou da raiz.

In [3]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

# Documentos reais e sintéticos ficam em subpastas distintas dentro de 01.arquivos_originais/ —
# só os reais passam por aqui (Gemini); os sintéticos (notebooks/00.gerar_documentos_sinteticos.ipynb)
# já nascem com gabarito conhecido, sem precisar de anotação via LLM.
PASTA_ENTRADA = RAIZ / "data" / "documento_social" / "01.arquivos_originais" / "arquivos_reais"
PASTA_SAIDA_BASE = RAIZ / "data" / "documento_social" / "02.gabarito_gemini"
PASTA_PROMPTS = RAIZ / "prompts"

print(f"Raiz do projeto: {RAIZ}")
print(f"Entrada:      {PASTA_ENTRADA}")
print(f"Saída (base): {PASTA_SAIDA_BASE}")
print(f"Prompts:      {PASTA_PROMPTS}")

Raiz do projeto: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3
Entrada:      d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\01.arquivos_originais\arquivos_reais
Saída (base): d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\02.gabarito_gemini
Prompts:      d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\prompts


## Carrega o prompt de sistema e o schema estruturado

Mantidos em `prompts/` para poderem ser versionados/editados sem mexer no código.

In [4]:
SYSTEM_INSTRUCTION = (PASTA_PROMPTS / "pii_ground_truth_system_prompt.txt").read_text(encoding="utf-8")
RESPONSE_SCHEMA = json.loads((PASTA_PROMPTS / "pii_ground_truth_response_schema.json").read_text(encoding="utf-8"))

print(SYSTEM_INSTRUCTION[:300], "...")

Você é um anotador de dados sensíveis (PII) em documentos brasileiros, para fins de \
pesquisa acadêmica sobre proteção de dados (LGPD).

Sua tarefa é identificar TODAS as ocorrências de:
- CPF (Cadastro de Pessoas Físicas). Inclua aqui também ocorrências de CIN (Carteira de
  Identidade Nacional, o ...


## Parâmetros do experimento

`MODEL` é o único parâmetro que normalmente muda entre execuções — trocar aqui para comparar `gemini-2.5-pro` vs `gemini-2.5-flash` vs `gemini-2.5-flash-lite` em uma amostra pequena antes de rodar o lote completo (ver anotacoes/anotacoes.txt, decisão sobre custo/qualidade).

In [5]:
MODEL = "gemini-2.5-flash"  # alternativas: "gemini-2.5-flash", "gemini-2.5-flash-lite"
SOBRESCREVER = False        # True para reprocessar arquivos que já têm gabarito salvo

EXTENSOES_SUPORTADAS = {".pdf", ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

# Uma subpasta por modelo, para poder comparar gabaritos de modelos diferentes lado a lado
# sem um sobrescrever o outro (ex.: gabarito_gemini/gemini-2.5-pro/, gabarito_gemini/gemini-2.5-flash-lite/).
PASTA_SAIDA = PASTA_SAIDA_BASE / MODEL
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print(f"Modelo: {MODEL}")
print(f"Saída:  {PASTA_SAIDA}")

Modelo: gemini-2.5-flash
Saída:  d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\02.gabarito_gemini\gemini-2.5-flash


## Tabela de preços (USD por 1M de tokens)

**IMPORTANTE:** preços da API do Gemini mudam com o tempo — os valores abaixo são uma referência e devem ser conferidos em https://ai.google.dev/pricing antes de rodar um lote grande, se o custo for uma preocupação real. Ajuste este dicionário se os preços tiverem mudado. `gemini-2.5-pro` tem faixa de preço maior para prompts acima de 200k tokens de entrada — não deve ser relevante para documentos de poucas dezenas de páginas, mas fica registrado o valor da faixa alta também.

In [6]:
# preço por 1.000.000 de tokens, em USD
PRECOS = {
    "gemini-2.5-pro": {"entrada": 1.25, "saida": 10.00, "entrada_acima_200k": 2.50, "saida_acima_200k": 15.00},
    "gemini-2.5-flash": {"entrada": 0.30, "saida": 2.50},
    "gemini-2.5-flash-lite": {"entrada": 0.10, "saida": 0.40},
}


def calcular_custo(model: str, tokens_entrada: int, tokens_saida: int) -> float:
    precos = PRECOS.get(model)
    if precos is None:
        return float("nan")
    preco_entrada = precos["entrada"]
    preco_saida = precos["saida"]
    if tokens_entrada > 200_000 and "entrada_acima_200k" in precos:
        preco_entrada = precos["entrada_acima_200k"]
        preco_saida = precos["saida_acima_200k"]
    custo = (tokens_entrada / 1_000_000) * preco_entrada + (tokens_saida / 1_000_000) * preco_saida
    return custo

In [7]:
import os
import sys

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    sys.exit("Defina GEMINI_API_KEY (variável de ambiente ou arquivo .env na raiz do projeto) antes de continuar.")

client = genai.Client(api_key=api_key)

## Função de processamento de um arquivo

**Nota sobre nomes de arquivo com acento:** o SDK do Gemini (File API) falha ao enviar arquivos cujo nome contém caracteres não-ASCII (`º`, `ª`, `ã`, `ç`, ...) — o cliente tenta codificar o nome do arquivo em ASCII em algum ponto da chamada HTTP e quebra. Como os arquivos reais deste projeto têm nomes assim, fazemos uma cópia temporária com nome seguro em ASCII antes de enviar, e apagamos a cópia depois.

A função retorna também as informações de uso de tokens (`usage_metadata` da resposta), usadas para calcular o custo de cada chamada.

In [8]:
def nome_ascii_seguro(nome: str) -> str:
    limpo = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    limpo = re.sub(r"[^A-Za-z0-9._-]+", "_", limpo).strip("_")
    return limpo or "documento"


def processar_arquivo(caminho: Path, model: str) -> tuple[dict, dict]:
    """Envia um arquivo ao Gemini via File API. Retorna (entidades, info_uso_tokens)."""
    nome_seguro = nome_ascii_seguro(caminho.stem) + caminho.suffix.lower()

    with tempfile.TemporaryDirectory() as tmp_dir:
        caminho_tmp = Path(tmp_dir) / nome_seguro
        shutil.copyfile(caminho, caminho_tmp)

        arquivo_remoto = client.files.upload(file=str(caminho_tmp))

        while arquivo_remoto.state.name == "PROCESSING":
            time.sleep(2)
            arquivo_remoto = client.files.get(name=arquivo_remoto.name)

    if arquivo_remoto.state.name == "FAILED":
        raise RuntimeError(f"Falha ao processar arquivo no Gemini: {caminho.name}")

    resposta = client.models.generate_content(
        model=model,
        contents=[
            arquivo_remoto,
            "Analise este documento e retorne todas as ocorrências de CPF, RG, CIN, NOME e ENDERECO "
            "conforme as instruções do sistema.",
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            temperature=0,
        ),
    )

    dados = json.loads(resposta.text)

    uso = resposta.usage_metadata
    tokens_entrada = getattr(uso, "prompt_token_count", 0) or 0
    tokens_saida = getattr(uso, "candidates_token_count", 0) or 0
    tokens_total = getattr(uso, "total_token_count", tokens_entrada + tokens_saida) or (tokens_entrada + tokens_saida)
    custo_usd = calcular_custo(model, tokens_entrada, tokens_saida)

    info_uso = {
        "tokens_entrada": tokens_entrada,
        "tokens_saida": tokens_saida,
        "tokens_total": tokens_total,
        "custo_usd": round(custo_usd, 5),
    }

    try:
        client.files.delete(name=arquivo_remoto.name)
    except Exception:
        pass

    return dados, info_uso

## Loop principal — processa todos os arquivos da pasta de entrada

In [9]:
import concurrent.futures

PARALELISMO = 10  # quantas requisições ao Gemini rodam em paralelo (baixe se começar a receber erro 429)

arquivos = sorted(
    p for p in PASTA_ENTRADA.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSOES_SUPORTADAS
)

if not arquivos:
    raise SystemExit(f"Nenhum arquivo suportado encontrado em {PASTA_ENTRADA}")

print(f"Encontrados {len(arquivos)} arquivo(s) em {PASTA_ENTRADA}")
print(f"Modelo: {MODEL} | Paralelismo: {PARALELISMO}")

arquivos_pendentes = []
for caminho in arquivos:
    destino = PASTA_SAIDA / f"{caminho.stem}.json"
    if destino.exists() and not SOBRESCREVER:
        print(f"{caminho.name}: já existe, pulando (SOBRESCREVER=True para refazer)")
        continue
    arquivos_pendentes.append(caminho)

print(f"\n{len(arquivos_pendentes)} arquivo(s) a processar.\n")


def processar_e_salvar(caminho: Path) -> dict:
    """Processa um arquivo e salva o resultado. Roda em uma thread do pool (I/O-bound: upload + espera do Gemini)."""
    destino = PASTA_SAIDA / f"{caminho.stem}.json"
    try:
        dados, info_uso = processar_arquivo(caminho, MODEL)
    except Exception as e:
        return {"caminho": caminho, "erro": str(e)}

    registro = {
        "document_id": caminho.stem,
        "source_file": caminho.name,
        "annotated_by": MODEL,
        "annotated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "usage": info_uso,
        "entities": dados.get("entities", []),
    }
    destino.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"caminho": caminho, "n_entidades": len(registro["entities"]), "info_uso": info_uso}


custo_total_usd = 0.0
concluidos = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=PARALELISMO) as executor:
    futuros = {executor.submit(processar_e_salvar, caminho): caminho for caminho in arquivos_pendentes}
    for futuro in concurrent.futures.as_completed(futuros):
        caminho = futuros[futuro]
        resultado = futuro.result()
        concluidos += 1

        if "erro" in resultado:
            print(f"[{concluidos}/{len(arquivos_pendentes)}] {caminho.name}: ERRO -> {resultado['erro']}")
            continue

        info_uso = resultado["info_uso"]
        custo_total_usd += info_uso["custo_usd"]
        print(
            f"[{concluidos}/{len(arquivos_pendentes)}] {caminho.name}: {resultado['n_entidades']} entidade(s) | "
            f"tokens entrada={info_uso['tokens_entrada']} saida={info_uso['tokens_saida']} | "
            f"custo=US$ {info_uso['custo_usd']:.5f}"
        )

print(f"\nConcluído. Custo total estimado desta execução: US$ {custo_total_usd:.4f}")

Encontrados 1010 arquivo(s) em d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\01.arquivos_originais\arquivos_reais
Modelo: gemini-2.5-flash | Paralelismo: 10
clpif4a2h00b92pqd67v27lm9-ALTERAÇÃO 1º DA EMPRESA TL TRANSPOSTES E LOGISTICA LTDA ME.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
clpif4mn700bb2pqde5cy04ej-INSTRUMENTO PARTICULAR 1º ALTERAÇÃO CONTRATUAL.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
clpif4v9800bd2pqd76oa1krz-INSTRUMENTO PARTICULAR 2º ALTERAÇÃO CONTRATUAL.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
clpif50i000bf2pqd2ky17hb4-INSTRUMENTO PARTICULAR 4 ALTERAÇÃO CONTRATUAL 11082010.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
clrpcwgv0003gj9qd31axe84h-2ª alteração contratual consolidada.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cls3l2zm400a3yrqdbj1yd5px-Ata out19 - inclusão objeto social.pdf.pdf: já existe, pulando (SOBRESCREVER=True para refazer)
cls3l36ct00a5y

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[1/505] clwrzli6y0019f9qd2pvl028a-6ª Alteração - Hollytec Comércio.pdf.pdf: 15 entidade(s) | tokens entrada=2577 saida=1722 | custo=US$ 0.00508
[2/505] clyx36b3d000evuqd2r1e9uf1-Grupo Marcolan Faturamento 05 2024 DDD.pdf.pdf: 9 entidade(s) | tokens entrada=1287 saida=721 | custo=US$ 0.00219
[3/505] cltejngnd00g4veqd560gajky-Alteração2 NC mar23.pdf.pdf: 33 entidade(s) | tokens entrada=2061 saida=3229 | custo=US$ 0.00869
[4/505] clzwp2gzh00a7ewqdarataogw-Alteração Contratual - F S Gulde (2).pdf.pdf: 23 entidade(s) | tokens entrada=2577 saida=2396 | custo=US$ 0.00676
[5/505] clyipaa73002kwxqdg8ei8dcq-4ª Alteração Contrato Social IPP - Arquivado (1).pdf.pdf: 13 entidade(s) | tokens entrada=2577 saida=1303 | custo=US$ 0.00403
[6/505] cm1jlgrib006udiqd6qar78sp-DOUGLAS AGUIAR - Certidao Simplificada.pdf.pdf: 5 entidade(s) | tokens entrada=1287 saida=476 | custo=US$ 0.00158
[7/505] clzue1kpc0014e8qd7oiz3k9b-Contrato - ALN2222379514-1.pdf.pdf: 44 entidade(s) | tokens entrada=2061 saida=4379 | c

## Conferência rápida do resultado (entidades e custo acumulado)

In [10]:
gabaritos = sorted(PASTA_SAIDA.glob("*.json"))
print(f"{len(gabaritos)} gabarito(s) salvo(s) em {PASTA_SAIDA}\n")

custo_acumulado = 0.0
for g in gabaritos:
    dados = json.loads(g.read_text(encoding="utf-8"))
    contagem = {}
    for e in dados["entities"]:
        contagem[e["type"]] = contagem.get(e["type"], 0) + 1
    custo = dados.get("usage", {}).get("custo_usd")
    if custo is not None:
        custo_acumulado += custo
    custo_str = f"US$ {custo:.5f}" if custo is not None else "N/A (gabarito antigo, sem usage)"
    print(f"{g.name}: {contagem} | custo: {custo_str}")

print(f"\nCusto acumulado (gabaritos com info de uso disponível): US$ {custo_acumulado:.4f}")

999 gabarito(s) salvo(s) em d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\02.gabarito_gemini\gemini-2.5-flash

clpif4a2h00b92pqd67v27lm9-ALTERAÇÃO 1º DA EMPRESA TL TRANSPOSTES E LOGISTICA LTDA ME.pdf.json: {'NOME': 27, 'CPF': 8, 'RG': 9, 'ENDERECO': 4} | custo: US$ 0.01268
clpif4mn700bb2pqde5cy04ej-INSTRUMENTO PARTICULAR 1º ALTERAÇÃO CONTRATUAL.pdf.json: {'NOME': 43, 'CPF': 10, 'RG': 10, 'ENDERECO': 8} | custo: US$ 0.01739
clpif4v9800bd2pqd76oa1krz-INSTRUMENTO PARTICULAR 2º ALTERAÇÃO CONTRATUAL.pdf.json: {'NOME': 18, 'CPF': 6, 'RG': 5, 'ENDERECO': 4} | custo: US$ 0.00910
clpif50i000bf2pqd2ky17hb4-INSTRUMENTO PARTICULAR 4 ALTERAÇÃO CONTRATUAL 11082010.pdf.json: {'NOME': 8, 'CPF': 3, 'RG': 3, 'ENDERECO': 3} | custo: US$ 0.00493
clrpcwgv0003gj9qd31axe84h-2ª alteração contratual consolidada.pdf.json: {'NOME': 27, 'CPF': 3, 'RG': 2, 'ENDERECO': 2} | custo: US$ 0.01106
cls3l2zm400a3yrqdbj1yd5px-Ata out19 - inclusão objeto social.pdf.json: {'NOME': 15} | custo: US$